In [27]:
from tqdm.notebook import tqdm
for i in tqdm(range(100)):
    pass

  0%|          | 0/100 [00:00<?, ?it/s]

In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
import mlflow

In [10]:

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp1")

<Experiment: artifact_location='/mlruns/1', creation_time=1774655207177, experiment_id='1', last_update_time=1774655207177, lifecycle_stage='active', name='exp1', tags={}, workspace='default'>

In [11]:
import polars as pl
data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

In [12]:
data_new = data_pl.with_columns(
    pl.col("含水率").log().alias("含水率_log")
)

In [16]:
data_new.tail(1)

sample number,species number,樹種,含水率,9993.76781,9989.9107,9986.05359,9982.19648,9978.33937,9974.48227,9970.62516,9966.76805,9962.91094,9959.05383,9955.19672,9951.33962,9947.48251,9943.6254,9939.76829,9935.91118,9932.05407,9928.19697,9924.33986,9920.48275,9916.62564,9912.76853,9908.91142,9905.05432,9901.19721,9897.3401,9893.48299,9889.62588,9885.76877,9881.91166,9878.05456,9874.19745,9870.34034,…,4134.82018,4130.96307,4127.10596,4123.24886,4119.39175,4115.53464,4111.67753,4107.82042,4103.96331,4100.10621,4096.2491,4092.39199,4088.53488,4084.67777,4080.82066,4076.96356,4073.10645,4069.24934,4065.39223,4061.53512,4057.67801,4053.82091,4049.9638,4046.10669,4042.24958,4038.39247,4034.53536,4030.67826,4026.82115,4022.96404,4019.10693,4015.24982,4011.39271,4007.5356,4003.6785,3999.82139,含水率_log
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1872,19,"""ホワイトオーク""",12.209302,-0.03632,-0.03638,-0.03664,-0.03696,-0.03705,-0.03699,-0.03693,-0.03681,-0.03673,-0.03679,-0.03683,-0.03683,-0.03697,-0.03718,-0.03728,-0.03734,-0.03741,-0.03752,-0.03775,-0.03797,-0.03803,-0.03806,-0.03818,-0.03834,-0.03844,-0.0384,-0.03822,-0.03809,-0.03807,-0.03805,-0.03807,-0.03807,-0.03813,…,0.29608,0.29757,0.29976,0.30277,0.30598,0.30891,0.31127,0.31272,0.31418,0.31718,0.32111,0.32444,0.32746,0.33114,0.33507,0.33846,0.34174,0.34542,0.34859,0.35068,0.35354,0.35835,0.36248,0.36393,0.3654,0.36865,0.37122,0.37217,0.37389,0.37569,0.37407,0.37107,0.37072,0.37242,0.37372,0.37127,2.502198


In [15]:
import polars as pl
import numpy as np
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# ===== MLflow設定 =====
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp2")

# ===== データ =====
pl_df = data_new  # ← あなたのデータ

# 目的変数
y = pl_df["含水率_log"].to_numpy()

# 不要列除外
drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]

X = pl_df.drop(drop_cols).to_numpy()

# ===== 分割 =====
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ===== 学習 =====
params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 5,
    "n_jobs": -1
}

with mlflow.start_run():

    # パラメータ記録
    mlflow.log_params(params)

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)

    # 予測
    pred = model.predict(X_valid)

    # RMSE（squared=False使えない環境対応）
    rmse = np.sqrt(mean_squared_error(y_valid, pred))

    # ログ
    mlflow.log_metric("rmse", rmse)

    # モデル保存
    mlflow.lightgbm.log_model(model, "model")

    print("RMSE:", rmse)

2026/04/02 21:23:43 INFO mlflow.tracking.fluent: Experiment with name 'exp2' does not exist. Creating a new experiment.


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.040501 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 396525
[LightGBM] [Info] Number of data points in the train set: 1057, number of used features: 1555
[LightGBM] [Info] Start training from score 3.408297


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/02 21:24:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


2026/04/02 21:24:28 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.18669948455892013
🏃 View run intrigued-lynx-633 at: http://mlflow:5000/#/experiments/5/runs/1879e468959d4faf8420fecbce8fe02f
🧪 View experiment at: http://mlflow:5000/#/experiments/5


In [18]:
import polars as pl
import numpy as np
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from ml_pipeline import FeatureEngineer

# ===== MLflow設定 =====
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp2")

# ===== データ =====
pl_df = data_new

# =========================
# 🟨先にsplit（重要）🟨
# =========================
train_df, valid_df = train_test_split(
    pl_df, test_size=0.2, random_state=42
)

# =========================
# 🟨Feature Engineering🟨
# =========================
fe = FeatureEngineer()

fe.fit(train_df)
train_df = fe.transform(train_df)
valid_df = fe.transform(valid_df)

# =========================
# 🟨yの取得位置変更🟨
# =========================
y_train = train_df["含水率_log"].to_numpy()
y_valid = valid_df["含水率_log"].to_numpy()

# 不要列除外
drop_cols = ["含水率", "含水率_log", "樹種", "sample number", "species number"]

# =========================
# 🟨Xの取得元変更🟨
# =========================
X_train = train_df.drop(drop_cols).to_numpy()
X_valid = valid_df.drop(drop_cols).to_numpy()

# ===== 学習 =====
params = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 64,
    "min_data_in_leaf": 5,
    "n_jobs": -1
}

with mlflow.start_run():

    mlflow.log_params(params)

    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    rmse = np.sqrt(mean_squared_error(y_valid, pred))
    mlflow.log_metric("rmse", rmse)

    mlflow.lightgbm.log_model(model, "model")

    print("RMSE:", rmse)

[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.201366 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 399075
[LightGBM] [Info] Number of data points in the train set: 1057, number of used features: 1565
[LightGBM] [Info] Start training from score 3.408297


/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/02 21:35:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5


2026/04/02 21:35:06 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RMSE: 0.11434015659032609
🏃 View run indecisive-cod-144 at: http://mlflow:5000/#/experiments/5/runs/f14e3f0d0ba34bfb95c3ebb657d1d298
🧪 View experiment at: http://mlflow:5000/#/experiments/5


### FEから一括で実行可能にした

In [39]:
import importlib
import ml_pipeline

importlib.reload(ml_pipeline)

/usr/local/lib/python3.11/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


<module 'ml_pipeline' from '/work/ml_pipeline.py'>

In [40]:

mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("exp2")

<Experiment: artifact_location='/mlruns/5', creation_time=1775165023203, experiment_id='5', last_update_time=1775165023203, lifecycle_stage='active', name='exp2', tags={}, workspace='default'>

In [41]:
import polars as pl
train_data_pl = pl.read_csv(r"../data/train.csv",encoding="shift_jis")

In [42]:
from ml_pipeline import MoisturePipeline, FullPipelineModel

pipe = MoisturePipeline()
rmse = pipe.fit(train_data_pl)

with mlflow.start_run():

    mlflow.log_metric("rmse", rmse)

    mlflow.pyfunc.log_model(
        name="model",
        python_model=FullPipelineModel(pipe)
    )

/usr/local/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
2026/04/02 21:52:04 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


🏃 View run sassy-newt-943 at: http://mlflow:5000/#/experiments/5/runs/32d52abc8e664911887d99540ea74a3f
🧪 View experiment at: http://mlflow:5000/#/experiments/5


### 予測ゾーン

In [21]:
submit_data_pl =  pl.read_csv(r"../data/test.csv",encoding="shift_jis")